# 01 · 环境与基准工具

这一章做三件事：确认 Colab 的 GPU 环境、建立后面九章都要用的测量工具、定义一个完全可控的 MiniGPT。

**为什么先建测量工具？**

推理 infra 的一切结论都建立在数字上。"延迟降低 40%"这句话如果没交代测量口径，就是废话——是在 batch=1 还是 batch=64 下测的？测的是单次调用还是含队排队的端到端？有没有做 warmup？

面试官问"这个数字怎么来的"时，能讲清口径的人，和只会背数字的人，是两种候选人。这一章就是为后者准备的。

**运行前检查**：Colab 菜单 → 代码执行程序 → 更改运行时类型 → 硬件加速器选 **GPU**。

In [ ]:
import sys
import platform

import torch

print(f"Python    : {sys.version.split()[0]}")
print(f"平台      : {platform.platform()}")
print(f"PyTorch   : {torch.__version__}")
print(f"CUDA 可用 : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU 型号  : {p.name}")
    print(f"显存      : {p.total_memory / 1024 ** 3:.1f} GB")
    print(f"计算能力  : sm_{p.major}{p.minor}")
    print(f"SM 数量   : {p.multi_processor_count}")
else:
    print()
    print("⚠️  没有检测到 GPU")
    print("   Colab：菜单 → 代码执行程序 → 更改运行时类型 → 硬件加速器 选 GPU")
    print("   第 01-08 章在 CPU 上也能跑完，只是慢；第 09、10 章必须要 GPU。")

In [ ]:
!nvidia-smi

## 一、引导单元：测量工具 + MiniGPT

下面这个单元格在每个 notebook 里都有一份完整副本，保证任何一章都能独立运行。

它包含五样东西：

1. **显卡规格 `CARD_SPECS` 和 `SPEC`**：你的卡有多少显存、多少带宽、多少算力。第 02、03 章直接拿它算账。
2. `sync / bench / peak_mem_mb`：测量工具。GPU 是**异步执行**的，不调 `torch.cuda.synchronize()` 就计时，测到的是 kernel 下发时间而不是执行时间——这是新手最常犯的测量错误。
3. `MiniGPT`：结构与 Llama 同源的因果语言模型，约 2700 万参数。
4. `generate_naive / generate_cached`：两条生成路径，用来对比有无 KV cache。
5. `kv_bytes`：KV cache 显存公式，第 03 章会实测验证它。

**关于 MiniGPT 的一个重要设计**：它的 `forward` 接受 `pos_offset` 参数，允许 KV cache 从任意位置继续。这正是实现连续批处理的前提——不同请求处在不同位置，调度器必须能把它们拼进同一个 batch。

In [ ]:
# ===== 引导单元：环境检查 + 测量工具 + MiniGPT（每章自带，直接运行）=====
# 说明：本单元在每个 notebook 里都有一份完整副本，目的是让任何一个 notebook
#       都能在 Colab 里零配置独立运行。想改模型结构，请改 tools/build_notebooks.py
#       里的 SETUP_CODE，然后重跑编译脚本。
#
# 架构对齐：下面这套推理核心刻意模仿了 vLLM V1 的模块划分与命名，
#   详见 docs/vllm-mapping.md 的对照表。
#       EngineCore.step()           ←→ vllm/v1/engine/core.py
#         ├─ Scheduler.schedule()   ←→ vllm/v1/core/sched/scheduler.py
#         ├─ ModelRunner.execute_model() ←→ vllm/v1/worker/gpu_model_runner.py
#         └─ Scheduler.update_from_output()
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# MiniGPT 只有 2700 万参数，用 float16 跑在 GPU 上；CPU 上 float16 很慢，用 float32
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# 常见卡的关键参数（近似值）。如果你的卡不在表里，直接在这里补一行：
#   "你的卡型号": {"mem_gb": .., "bw_gbps": .., "fp16_tflops": .., "arch": ".."},
# 三个数字都能在厂商 datasheet 上查到。第 02、03 章会用到它们。
CARD_SPECS = {
    "Tesla T4":        {"mem_gb": 16, "bw_gbps": 320,  "fp16_tflops": 65,  "arch": "Turing sm75"},
    "Tesla V100":      {"mem_gb": 16, "bw_gbps": 900,  "fp16_tflops": 125, "arch": "Volta sm70"},
    "A100-SXM4-40GB":  {"mem_gb": 40, "bw_gbps": 1555, "fp16_tflops": 312, "arch": "Ampere sm80"},
    "A100-SXM4-80GB":  {"mem_gb": 80, "bw_gbps": 2039, "fp16_tflops": 312, "arch": "Ampere sm80"},
    "L4":              {"mem_gb": 24, "bw_gbps": 300,  "fp16_tflops": 121, "arch": "Ada sm89"},
    "A10G":            {"mem_gb": 24, "bw_gbps": 600,  "fp16_tflops": 125, "arch": "Ampere sm86"},
    "H100 PCIe":       {"mem_gb": 80, "bw_gbps": 2000, "fp16_tflops": 756, "arch": "Hopper sm90"},
    "H100 80GB HBM3":  {"mem_gb": 80, "bw_gbps": 3350, "fp16_tflops": 989, "arch": "Hopper sm90"},
}


def lookup_card():
    """按 GPU 名称匹配规格表。匹配不到就返回零值，提醒你手工补。"""
    if not torch.cuda.is_available():
        return {"name": "CPU", "mem_gb": 0, "bw_gbps": 0, "fp16_tflops": 0, "arch": "CPU"}
    name = torch.cuda.get_device_properties(0).name
    for key, spec in CARD_SPECS.items():
        # 双向包含匹配：Colab 可能报 "Tesla T4"，也可能报 "NVIDIA L4"
        if key.lower() in name.lower() or name.lower().replace("nvidia ", "") in key.lower():
            return {"name": name, **spec}
    return {
        "name": name,
        "mem_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 1),
        "bw_gbps": 0,
        "fp16_tflops": 0,
        "arch": "未知卡型 → 请查 datasheet 后补进 CARD_SPECS",
    }


SPEC = lookup_card()


def sync():
    """GPU 是异步执行的，计时前必须同步，否则测到的是下发时间不是执行时间。"""
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def bench(fn, warmup=3, iters=10):
    """返回单次调用的平均耗时（毫秒）。warmup 用来排除首次 kernel 编译等开销。"""
    for _ in range(warmup):
        fn()
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    sync()
    return (time.perf_counter() - t0) / iters * 1000.0


def peak_mem_mb():
    """当前 CUDA 峰值显存占用（MB）。"""
    if DEVICE != "cuda":
        return 0.0
    return torch.cuda.max_memory_allocated() / 1024 ** 2


def reset_peak():
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()


class Config:
    def __init__(self, vocab_size=50257, block_size=1024, n_layer=4, n_head=6, n_embd=384):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head


class CausalSelfAttention(nn.Module):
    """因果自注意力，支持 KV cache。

    past_kv 传入历史的 (k, v)，本步只为新 token 计算 Q/K/V，然后拼在历史后面。
    返回 (输出, 更新后的 (k, v))，其中 k/v 的 shape 是 (B, n_head, 总长度, head_dim)。
    """

    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.head_dim
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x, past_kv=None, attn_mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)

        S = k.size(2)  # 总长度 = 历史 + 本步新增
        if attn_mask is None:
            # 默认因果掩码：本步第 i 个 query 的绝对位置是 S-T+i，只能看见 <= 它的 key
            mask = torch.ones(T, S, device=x.device).tril(diagonal=S - T).bool()
        else:
            # 外部传入的掩码，用于一个 batch 里混合不同进度的序列（第 04、06 章）
            mask = attn_mask
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), (k, v)


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x, past_kv=None, attn_mask=None):
        h, present = self.attn(self.ln_1(x), past_kv, attn_mask)
        x = x + h
        x = x + self.mlp(self.ln_2(x))
        return x, present


class MiniGPT(nn.Module):
    """极简 GPT，结构与 Llama 同源：pre-norm + 因果注意力 + 4 倍扩张 MLP + 权重共享。

    与 Llama 的两处差异：
      - 用可学习位置编码代替 RoPE（简化实现，不影响调度实验的结论）
      - 没有 GQA（本仓库是 MHA，第 03 章会手工比较两者的 KV cache 大小）
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # 权重共享，省一份 embedding 参数

        def init(m):
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

        self.apply(init)

    def forward(self, idx, past_kvs=None, pos_offset=0, attn_mask=None):
        """idx: (B, T) 的 token id。

        past_kvs: 长度等于层数的列表，每项是 (k, v)；None 表示从零开始（prefill）。
        pos_offset: 本次输入的第一个 token 的绝对位置。传 int 表示整个 batch 用同一个
                    偏移；传 shape (B,) 的张量表示每条序列各用各的偏移——当 batch 里
                    混合了不同进度的请求时必须这样传。
        attn_mask: 可选的自定义注意力掩码，用于屏蔽填充位。
        """
        B, T = idx.shape
        if torch.is_tensor(pos_offset):
            pos = pos_offset.view(B, 1) + torch.arange(T, device=idx.device)[None, :]
        else:
            pos = torch.arange(pos_offset, pos_offset + T, device=idx.device)[None, :].expand(B, T)
        x = self.wte(idx) + self.wpe(pos)

        presents = []
        for i, blk in enumerate(self.blocks):
            past = None if past_kvs is None else past_kvs[i]
            x, present = blk(x, past, attn_mask)
            presents.append(present)
        return self.lm_head(self.ln_f(x)), presents

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def build_model(seed=0, device=DEVICE, dtype=DTYPE, **kw):
    torch.manual_seed(seed)
    cfg = Config(**kw)
    model = MiniGPT(cfg).to(device=device, dtype=dtype)
    return model.eval()


@torch.no_grad()
def generate_naive(model, idx, max_new_tokens):
    """不用 KV cache：每一步都把完整序列重新算一遍（O(n^2) 重算）。"""
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -model.cfg.block_size:])
        idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return idx


@torch.no_grad()
def generate_cached(model, idx, max_new_tokens):
    """用 KV cache：prompt 只 prefill 一次，之后每步只喂 1 个 token。"""
    logits, past = model(idx)
    nxt = logits[:, -1].argmax(-1, keepdim=True)
    out = [nxt]
    pos = idx.size(1)
    for _ in range(max_new_tokens - 1):
        logits, past = model(nxt, past_kvs=past, pos_offset=pos)
        pos += 1
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        out.append(nxt)
    return torch.cat([idx] + out, dim=1)


def kv_bytes(n_layer, n_kv_head, head_dim, seq_len, batch=1, dtype_bytes=2):
    """KV cache 字节数。注意是 2（K 和 V 各一份）。"""
    return 2 * n_layer * n_kv_head * head_dim * seq_len * batch * dtype_bytes


# ========== 以下是模仿 vLLM V1 架构的推理核心 ==========


class Request:
    """对应 vllm/v1/request.py 的 Request。

    num_computed_tokens 是 vLLM 里最核心的一个字段：它记录这条请求已经有
    多少 token 的 KV 被算过。prefill、chunked prefill、前缀缓存命中——
    三种看起来完全不同的场景，在 vLLM 里都只是「把 num_computed_tokens 往前推」。
    理解这一点，chunked prefill 就不再是独立机制，而是这个字段的自然结果。
    """

    def __init__(self, request_id, prompt_token_ids, max_tokens):
        self.request_id = request_id
        self.prompt_token_ids = list(prompt_token_ids)
        self.max_tokens = max_tokens
        self.output_token_ids = []
        self.num_computed_tokens = 0
        self.status = "waiting"      # waiting / running / finished
        # 本仓库简化：直接把 KV 张量挂在请求上。
        # 真实 vLLM 不这么做——请求只持有 block_table，物理 block 由 KVCacheManager 管（第 05 章）。
        self.past = None

    @property
    def num_prompt_tokens(self):
        return len(self.prompt_token_ids)

    def all_token_ids(self):
        return self.prompt_token_ids + self.output_token_ids

    def num_tokens_to_schedule(self):
        """还欠多少 token 没算：prefill 阶段是剩余 prompt 长度，decode 阶段是 1。"""
        if self.num_computed_tokens < self.num_prompt_tokens:
            return self.num_prompt_tokens - self.num_computed_tokens
        return 1

    @property
    def is_finished(self):
        return len(self.output_token_ids) >= self.max_tokens

    def __repr__(self):
        return (f"Request({self.request_id}, computed={self.num_computed_tokens}"
                f"/{self.num_prompt_tokens}, out={len(self.output_token_ids)}"
                f"/{self.max_tokens}, {self.status})")


class SchedulerOutput:
    """对应 vllm/v1/core/sched/output.py 的 SchedulerOutput。

    调度与执行之间唯一的接口。真实 vLLM 里这个结构还包含 block 分配结果、
    抢占列表等字段，这里只保留最必要的两个。
    """

    def __init__(self, scheduled_reqs, num_scheduled_tokens):
        self.scheduled_reqs = scheduled_reqs
        self.num_scheduled_tokens = num_scheduled_tokens   # {request_id: n}

    def __len__(self):
        return len(self.scheduled_reqs)


class Scheduler:
    """对应 vllm/v1/core/sched/scheduler.py 的 Scheduler。

    职责边界是这个架构里最值得学的一点：Scheduler 只决定
    「这一轮跑哪些请求、各自跑几个 token」，它既不碰显存也不碰模型。

        显存分配 → KVCacheManager（第 05 章）
        真正计算 → ModelRunner

    三个模块分离，才能各自独立替换实现。面试被问「说说 vLLM 的架构」时，
    先把这个职责划分讲清楚，比背模块名有用得多。
    """

    def __init__(self, max_num_seqs=8, max_num_batched_tokens=2048):
        self.waiting = []
        self.running = []
        self.finished = []
        self.max_num_seqs = max_num_seqs
        # 这个预算就是 chunked prefill 的开关：调小它，长 prompt 自然被切成多轮（第 06 章）
        self.max_num_batched_tokens = max_num_batched_tokens
        self.step_id = 0

    def add_request(self, req):
        self.waiting.append(req)

    def has_unfinished(self):
        return bool(self.waiting or self.running)

    def schedule(self):
        scheduled, num_tokens = [], {}
        budget = self.max_num_batched_tokens

        # 第一优先：正在跑的请求。已进 decode 的排 1 个 token；
        # 还在做 chunked prefill 的按剩余量排，但受 budget 限制。
        for req in list(self.running):
            if budget <= 0 or len(scheduled) >= self.max_num_seqs:
                break
            n = min(req.num_tokens_to_schedule(), budget)
            scheduled.append(req)
            num_tokens[req.request_id] = n
            budget -= n

        # 第二优先：从队列里补新请求进来做 prefill
        for req in list(self.waiting):
            if budget <= 0 or len(scheduled) >= self.max_num_seqs:
                break
            n = min(req.num_tokens_to_schedule(), budget)
            scheduled.append(req)
            num_tokens[req.request_id] = n
            budget -= n
            self.waiting.remove(req)
            req.status = "running"
            self.running.append(req)

        self.step_id += 1
        return SchedulerOutput(scheduled, num_tokens)

    def update_from_output(self, sched_out, sampled):
        """对应 vLLM 的 update_from_output：写回采样结果，处理完成与回收。

        本轮被调度但没产生 token 的请求（比如 chunked prefill 的中间块）
        不会出现在 sampled 里，它们保持 running，下一轮继续。
        """
        for req in sched_out.scheduled_reqs:
            if req.request_id not in sampled:
                continue
            req.output_token_ids.append(sampled[req.request_id])
            if req.is_finished:
                req.status = "finished"
                if req in self.running:
                    self.running.remove(req)
                self.finished.append(req)
                req.past = None      # 简化回收；真实 vLLM 走 KVCacheManager.free()


class ModelRunner:
    """对应 vllm/v1/worker/gpu_model_runner.py 的 GPUModelRunner。

    职责：把 Scheduler 排好的一批请求拼成一次前向，返回新采样的 token。

    与真实 vLLM 的差距（要如实知道）：
      · vLLM 用 block_table 让每条序列的 KV 物理上不连续，所以不需要填充；
        这里用「右填充 + 逐序列掩码」对齐，会浪费显存——第 05 章解决。
      · vLLM 会把 prefill 和 decode 混在同一个 batch 里跑；这里分成两组处理，
        纯粹是为了让代码可读，结论不受影响。
      · 输入准备、CUDA graph、attention metadata 这些都被省掉了。
    """

    def __init__(self, model):
        self.model = model

    @torch.no_grad()
    def _run_decode_batch(self, reqs):
        """把一批进度不同的 decode 请求拼成一次前向。"""
        B = len(reqs)
        lens = [r.num_computed_tokens for r in reqs]
        Lmax = max(lens)
        n_layer = self.model.cfg.n_layer

        padded = []
        for layer in range(n_layer):
            ks, vs = [], []
            for r in reqs:
                k, v = r.past[layer]
                pad = Lmax - k.size(2)
                if pad:
                    k = F.pad(k, (0, 0, 0, pad))
                    v = F.pad(v, (0, 0, 0, pad))
                ks.append(k)
                vs.append(v)
            padded.append((torch.cat(ks, 0), torch.cat(vs, 0)))

        # 逐序列掩码：真实历史 [0, L_i) + 新 token 落在下标 Lmax
        S = Lmax + 1
        mask = torch.zeros(B, 1, 1, S, dtype=torch.bool, device=DEVICE)
        for i, r in enumerate(reqs):
            mask[i, 0, 0, : lens[i]] = True
            mask[i, 0, 0, Lmax] = True

        ids = torch.tensor([[r.all_token_ids()[r.num_computed_tokens]] for r in reqs],
                           device=DEVICE)
        pos = torch.tensor(lens, device=DEVICE)
        logits, past = self.model(ids, past_kvs=padded, pos_offset=pos, attn_mask=mask)

        sampled = {}
        for i, r in enumerate(reqs):
            rebuilt = []
            for layer in range(n_layer):
                k_all, v_all = past[layer]
                k = torch.cat([k_all[i:i + 1, :, : lens[i]],
                               k_all[i:i + 1, :, Lmax:Lmax + 1]], dim=2)
                v = torch.cat([v_all[i:i + 1, :, : lens[i]],
                               v_all[i:i + 1, :, Lmax:Lmax + 1]], dim=2)
                rebuilt.append((k, v))
            r.past = rebuilt
            r.num_computed_tokens += 1
            sampled[r.request_id] = int(logits[i, -1].argmax(-1).item())
        return sampled

    @torch.no_grad()
    def execute_model(self, sched_out):
        decode_reqs, prefill_reqs = [], []
        for r in sched_out.scheduled_reqs:
            # 判断依据是「prompt 算完了没有」，而不是「本轮排了几个 token」
            if r.num_computed_tokens >= r.num_prompt_tokens:
                decode_reqs.append(r)
            else:
                prefill_reqs.append(r)

        sampled = {}
        if decode_reqs:
            sampled.update(self._run_decode_batch(decode_reqs))

        for r in prefill_reqs:
            n = sched_out.num_scheduled_tokens[r.request_id]
            start = r.num_computed_tokens
            chunk = r.all_token_ids()[start:start + n]
            toks = torch.tensor([chunk], device=DEVICE)
            logits, past = self.model(toks, past_kvs=r.past, pos_offset=start)
            r.past = past
            r.num_computed_tokens += len(chunk)
            # 只有 prompt 全部算完，才能采样第一个输出 token
            if r.num_computed_tokens >= r.num_prompt_tokens:
                sampled[r.request_id] = int(logits[:, -1].argmax(-1).item())
        return sampled


class EngineCore:
    """对应 vllm/v1/engine/core.py 的 EngineCore。

    整个 vLLM 的推理服务就跑在这三步上：

        schedule()            决定这一轮跑什么
        execute_model()       跑模型
        update_from_output()  把结果写回请求状态

    读懂这个循环你就抓住了 vLLM 的主干。后面所有优化——chunked prefill、
    前缀缓存、抢占、投机解码——都是在这三步里插桩。
    """

    def __init__(self, model, scheduler=None):
        self.scheduler = scheduler or Scheduler()
        self.runner = ModelRunner(model)
        self.step_id = 0
        self.steps = 0

    def step(self):
        sched_out = self.scheduler.schedule()
        if len(sched_out) == 0:
            return None
        sampled = self.runner.execute_model(sched_out)
        self.scheduler.update_from_output(sched_out, sampled)
        self.step_id += 1
        self.steps += 1
        return sampled

    def run(self, max_steps=10000):
        while self.scheduler.has_unfinished() and self.steps < max_steps:
            self.step()
        return self.steps


print(f"引导单元加载完成 | device={DEVICE} dtype={DTYPE} torch={torch.__version__}")
# ===== 引导单元结束 =====

## 二、你的卡是什么规格

后面每一章都要用这三个数字：

| 参数 | 含义 | 为什么重要 |
|---|---|---|
| 显存容量 | 能装多少权重 + KV cache | 第 03 章算最大并发 |
| 显存带宽 | 每秒能从显存搬多少字节 | **decode 阶段的瓶颈就在这里** |
| FP16 算力 | 每秒能做多少次浮点运算 | **prefill 阶段的瓶颈** |

把「算力 ÷ 带宽」算出来，你就有了判断一个操作是 compute-bound 还是 memory-bound 的标尺。第 02 章会用到。

> 表里是常见卡的近似规格（FP16 稠密算力，不含稀疏加速）。**你的卡不在表里的话，直接在上面那个引导单元里补一行**，三个数字在厂商 datasheet 上都能查到。

In [ ]:
for k, v in SPEC.items():
    print(f"{k:12s}: {v}")

if SPEC["bw_gbps"]:
    ratio = SPEC["fp16_tflops"] * 1e12 / (SPEC["bw_gbps"] * 1e9)
    print(f"\n算力/带宽比 = {ratio:.0f} FLOP/byte")
    print("→ 记住这个数，第 02 章用它判断 prefill 和 decode 各自的瓶颈。")

## 三、认识你的实验对象

MiniGPT 的参数配置：4 层、6 个注意力头、隐藏维度 384、词表 50257、上下文 1024。

这个规模是刻意选的：它小到能在 T4 上秒级完成任务，又大到足以让显存带宽和调度开销呈现出和真实大模型相同的规律。**我们要观察的是比例关系，不是绝对值。**

In [ ]:
model = build_model()

print(f"结构      : {model.cfg.n_layer} 层 / {model.cfg.n_head} 头 / head_dim={model.cfg.head_dim}")
print(f"参数量    : {model.n_params / 1e6:.1f} M")
print(f"权重显存  : {model.n_params * 2 / 1024 ** 2:.1f} MB (fp16)")

reset_peak()
_ = model(torch.randint(0, model.cfg.vocab_size, (1, 128), device=DEVICE))
print(f"跑一次 128 token 的峰值显存: {peak_mem_mb():.1f} MB")

## 四、第一次测量：prefill 一次 vs 逐 token 生成

两种操作的计算形态完全不同：

- **prefill**：一次性把整段 prompt 喂进去，是一大块稠密矩阵乘法。批量大、并行度高。
- **decode**：一次只生成一个 token，每步都要把**全部权重**从显存读一遍。批量小、访存密集。

先建立直觉，第 02 章会定量分析。

In [ ]:
B, T = 8, 512
idx = torch.randint(0, model.cfg.vocab_size, (B, T), device=DEVICE)

reset_peak()
ms = bench(lambda: model(idx), warmup=3, iters=10)
tokens = B * T
print(f"prefill {B} 条 × {T} token")
print(f"  耗时      : {ms:.1f} ms")
print(f"  吞吐      : {tokens / (ms / 1000):,.0f} token/s")
print(f"  峰值显存  : {peak_mem_mb():.0f} MB")

In [ ]:
prompt = torch.randint(0, model.cfg.vocab_size, (1, 64), device=DEVICE)
N = 32

ms_naive = bench(lambda: generate_naive(model, prompt, N), warmup=1, iters=3)
ms_cached = bench(lambda: generate_cached(model, prompt, N), warmup=1, iters=3)

print(f"生成 {N} 个 token（prompt 长度 64）")
print(f"  不用 KV cache : {ms_naive:8.1f} ms")
print(f"  使用 KV cache : {ms_cached:8.1f} ms")
print(f"  加速比        : {ms_naive / ms_cached:.2f}x")

### 顺手做一个正确性验证

性能优化最怕的是"变快了但算错了"。`generate_cached` 用 KV cache 只算新 token 的 Q/K/V，`generate_naive` 每步重算全部——**两者的输出必须逐位相同**。

这个习惯要带到第 05 章：验证前缀复用时，同样用"逐位比对 logits"而不是"看起来差不多"。

In [ ]:
a = generate_naive(model, prompt, N)
b = generate_cached(model, prompt, N)
print("两条路径输出完全一致:", torch.equal(a, b))

## 五、小结与作业

**本章产出**：一个可复用的测量框架，一个可控的模型，以及一条正确的性能验证方法。

三个容易踩的坑，后面每一章都会反复遇到：

1. **忘记 `torch.cuda.synchronize()`** → 测出来是下发时间，数字好看得离谱。
2. **忘记 warmup** → 第一次调用包含 kernel 编译、显存分配器预热，数字难看。
3. **只看平均值不看分布** → 推理 infra 里 p99 往往比均值更重要，第 06 章会专门处理尾延迟。

**作业（做完再进第 02 章）**

1. 把 `CARD_SPECS` 里你的卡补全，手算「算力 ÷ 带宽」得到算力带宽比。
2. 把 `bench()` 的 `iters` 从 10 改成 100，观察数字是否稳定；如果不稳定，想想是什么在干扰（提示：Colab 是共享实例）。
3. 把 MiniGPT 改成 `n_layer=8`，重复上面的测量，看看参数量翻倍对 prefill 和 decode 的影响是否一样。

**下一章**：用刚才记下的算力带宽比，定量分析 prefill 和 decode 各自的瓶颈，并实测验证。